# CMDF feature extraction — cross-modal differential features for 1p/19q codeletion

This notebook replaces `feature_extraction_cmdf (1).ipynb`. What changed and why:

| Original notebook | This version | Why |
|---|---|---|
| Google Colab + Drive paths, UTSW only | local/WSL paths; UTSW **and** UCSF-PDGM from `1P19Q -Dataset` | the current dataset |
| every patient with a 1p/19q label (318, incl. IDH-wildtype) | IDH-mutant, label resolved, imaging QC passed (150) — the habitat study's cohort | IDH-wildtype tumours are never codeleted, so including them makes the task trivially easy and the AUROC meaningless for the clinical question |
| T1-Gd vs T2 only | all 4 sequences, all 6 pairs | the T2–FLAIR pair is the one with a known radiological sign |
| 16³ patches, re-extracted | the habitat study's cached 32³ patches (N4, brain z-score, 93 features/sequence) | identical preprocessing for both studies, no duplicated 35-min extraction |
| `variance` metric = (a−b)²/4 | dropped | an exact function of the L2 metric |
| `outlier_deviation` σ over **all** patients | cosine profile concordance with **training-fold** scaling | the original σ leaked test patients into features |
| cosine on raw features | cosine after per-sequence standardisation | raw cosine is dominated by large-valued features (Energy ≈ 10⁴) |
| — | voxel-level CMDF and a quantitative **T2–FLAIR mismatch** block | anchors CMDF to an established imaging marker |

The analysis plan is fixed in `../PROTOCOL.md`.

In [ ]:
# Run inside WSL with the habitat study's environment (~/venvs/habitat2; add ipykernel to it to use this notebook).
# Paths default to the workspace layout; override with GLIOMA_DATA_ROOT, HABITAT_OUTPUT_DIR and CMDF_OUTPUT_DIR.
import sys
from pathlib import Path
CMDF_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CMDF_DIR))

import pandas as pd
from IPython.display import Image, Markdown, display
from cmdf1p19q.config import build_cmdf_config
from cmdf1p19q.study import CMDFStudy

hcfg, ccfg = build_cmdf_config()
study = CMDFStudy(hcfg, ccfg)
study.protocol_sha = __import__("habitat1p19q.reporting", fromlist=["sha256"]).sha256(CMDF_DIR / "PROTOCOL.md")
print("data:", hcfg["data_root"], "| habitat cache:", hcfg["output_dir"], "| outputs:", ccfg["output_dir"])

## 1. Cohort, imaging QC and the cached patch features

In [ ]:
study.stage_data()

## 2. Voxel-level CMDF and the T2–FLAIR mismatch block (N4 + z-score, cached per patient)

In [ ]:
study.stage_voxel()
study.V.describe().T.head(12)

## 3. Feature blocks P (single sequence), D (patch CMDF), V/MM (voxel CMDF)

In [ ]:
study.stage_blocks()
Ftr, Fte, _ = study.builder.build(study.ids["UTSW"], [])
from cmdf1p19q.features import model_columns
pd.Series({g: len(model_columns(g, Ftr)) for g in ["P", "D", "D_T2FL", "V", "MM", "M2"]}, name="columns")

In [ ]:
# Example: the T2–FLAIR block for three patients
Ftr[[c for c in model_columns("D_T2FL", Ftr)][:6] + ["MM__mismatch_frac"]].head(3)